> **Kept for reference.** This is the v3 model, replaced by `ClusteringModel.ipynb` (v4). It measures every distance in a straight line.

# Two-layer cabinet clustering → daily clusters

**Purpose.** Group cabinets into **clusters**. Each cluster is one day of field work for one **team of up to 3 people**
and becomes a card on the manager's calendar. The model does **not** assign workers; the manager decides which
team takes which card.

**Why 12–18 tickets/day.** One person can handle about 6 tickets a day, so a team of 3 can handle 3 × 6 = **18**
(`CAP`). A person also needs at least 4 tickets for a worthwhile day, so a team of 3 needs 3 × 4 = **12**
(`MIN_DEMAND`).

**Smaller teams.** In a quiet area there may not be 12 tickets/day within `RADIUS_KM`. Such a cluster gets a smaller
team instead, down to one person. Every team size uses the same two numbers per person (at least 4, at most 6):

| Team | Minimum | Maximum |
|---|---|---|
| 3 people | 12 | 18 |
| 2 people | 8 | 12 |
| 1 person | 4 | 6 (7 as a special case) |

The team always follows the demand: a cluster of 5 tickets/day gets 1 person, one of 9 gets 2. There is no team of 3
under 12, no team of 2 under 8 and no team of 1 under 4. One person may take up to `SOLO_MAX_DEMAND` (7) only as a
**special case**, when the cluster can't get down to 6.

**The three rules every cluster must satisfy**

1. **Demand** – the cluster's demand (expected tickets per day) is between the minimum and the maximum of its
   team: `MIN_DEMAND` (12) to `CAP` (18) for a full team.
2. **Distance** – every cabinet is within `RADIUS_KM` (7.5 km) of the cluster centre, in a straight line. So a
   cluster is at most 15 km from end to end.
3. **No overlapping outlines** – draw a rubber band around each cluster's cabinets (its *outline*); no two
   outlines may overlap. So no cluster sits inside or pokes through another one.

**Leftovers.** A cluster that is too small even for one person is broken up at the very end: each of its cabinets
joins the cluster with the nearest border and is flagged as a *leftover* in the outputs, so the manager can see it.
Every cabinet always ends up in a cluster. A leftover counts for its cluster's demand and team, but it does not move
the cluster centre and is drawn apart on the map.

**How we get there**

| Step | What it does |
|---|---|
| Data check | Load the cabinets and make sure no single cabinet has more than a full day's demand on its own. |
| Demand Layer | Work out how many clusters are needed and share the cabinets out between them, so every cluster is ≤ `CAP` and the demand is spread evenly. |
| Distance Layer | Check every cabinet is within `RADIUS_KM` of its cluster centre. A cabinet that is too far is moved to a neighbouring cluster, taken out on its own, or its cluster is split in two. |
| Minimum demand | Bring every cluster below `MIN_DEMAND` up to it: merge it with a neighbour, share its cabinets out, or take over cabinets from neighbours. |
| Smaller teams | A cluster still below `MIN_DEMAND` is tried with a team of 2 (8–12/day), then a team of 1 (4–6/day). A cluster between two bands (7 to 8/day) passes cabinets on or takes some over. |
| Leftovers | A cluster no team fits is broken up: each of its cabinets joins the cluster with the nearest border. |
| Secondary cabinets | For every cluster, list the nearest 10 cabinets of *other* clusters that are also within `RADIUS_KM` (straight line) of its centre – nearby backup work, not counted in demand. |
| Validation | Hard checks – the notebook stops with an error if any rule is broken. |
| Outputs | CSV files (clusters, teams, cabinet-to-cluster assignments, secondaries) and an interactive map. |

Every step keeps all three rules: no change is made if it would push a cluster over its maximum, put a cabinet more than
`RADIUS_KM` from its centre, or make two outlines overlap.

**Data.** The file named by `DATA_FILE` in the parameters cell (by default the mock file `Dataset/mock_cabinets.csv`).
It needs the columns `CabinetId, cabinetlat, cabinetlong, total_tickets, total_days, demand`.
`demand` = `total_tickets / total_days`, i.e. average tickets per day, and is used as-is.

**Distance.** All distances are **haversine** km: the straight-line ("as the crow flies") distance over the earth's
surface. No road data is needed. Real driving distances are longer.

## Setup and parameters
The input file and all tunable numbers live in the next cell and are used everywhere downstream. To run on new data,
change `DATA_FILE` (path relative to the project root) and run the whole notebook.

In [ ]:
DATA_FILE = "Dataset/mock_cabinets.csv"  # input file, relative to the project root

PEOPLE_PER_CLUSTER = 3                   # biggest team: every cluster starts as one day for a team this size
MIN_PEOPLE = 1                           # smallest team a cluster can be given (never below 1)
DEMAND_PER_PERSON = 6                    # most tickets/day one person can handle
MIN_DEMAND_PER_PERSON = 4                # fewest tickets/day that make a worthwhile day for one person
SOLO_MAX_DEMAND = 7                      # special case: one person may take up to this, only if the cluster can't
                                         # get down to DEMAND_PER_PERSON
CAP = PEOPLE_PER_CLUSTER * DEMAND_PER_PERSON              # max tickets/day per cluster (18)
MIN_DEMAND = PEOPLE_PER_CLUSTER * MIN_DEMAND_PER_PERSON   # min tickets/day for a full team (12)
RADIUS_KM = 7.5                          # every cabinet within this straight-line distance of its cluster centre
                                         # (so a cluster is at most 15 km across)
MAX_SECONDARY = 10                       # secondary cabinets kept per cluster: the nearest ones (None = keep all)

SEED = 67
BALANCE_TOL = 1.0     # Demand Layer: stop evening out once heaviest - lightest cluster <= this (tickets/day)

In [ ]:
# Install any missing library into the Python that is running this notebook (e.g. Anaconda),
# so "Run All" works on a fresh machine. Already-installed libraries are left alone.
import importlib.util
import subprocess
import sys

REQUIRED = {"pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "folium": "folium",
            "shapely": "shapely"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Python", sys.version.split()[0], "- all required libraries available")

In [ ]:
import math
import os
import time
from pathlib import Path

# One thread for k-means: avoids a scikit-learn memory-leak warning on Windows/Anaconda (MKL).
# Must be set before numpy/scikit-learn are imported; the data is small, so speed is unaffected.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import folium
import numpy as np
import pandas as pd
import shapely
from shapely.geometry import mapping
from sklearn.cluster import KMeans

np.random.seed(SEED)

# The notebook may be executed from the project root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name in ("notebooks", "previous_versions") else Path.cwd()
DATA_PATH = ROOT / DATA_FILE
OUT_DIR = ROOT / "outputs" / "v3"
OUT_DIR.mkdir(exist_ok=True)

EARTH_RADIUS_KM = 6371.0088
TOL = 1e-6          # numerical tolerance for demand sums
START = time.time()
pd.set_option("display.width", 140)

## Load and check the data
We read the cabinets file (it is never modified) and look at the overall demand. Cabinets with zero demand (no
tickets in the period) need no visits, so they are skipped with a printed warning instead of stopping the notebook.

**Safety check.** A cluster is one day of work, so a single cabinet with demand above `CAP` could never fit into
any cluster. Our data never has such a cabinet, so this check should never trigger: it only guards against
unexpected data, and stops the notebook with a clear message if it happens.

**How many clusters?** The fewest possible is `ceil(total demand / 18)`: every cluster a full team at `CAP`. The
most is `floor(total demand / 4)`: every cluster a one-person day at its minimum. The closer to the fewest, the
fewer days of work there are to plan.

In [ ]:
assert 1 <= MIN_PEOPLE <= PEOPLE_PER_CLUSTER, "MIN_PEOPLE must be between 1 and PEOPLE_PER_CLUSTER"
assert 0 < MIN_DEMAND_PER_PERSON <= DEMAND_PER_PERSON, "MIN_DEMAND_PER_PERSON must be between 0 and DEMAND_PER_PERSON"

cabinets = pd.read_csv(DATA_PATH)
assert cabinets["CabinetId"].is_unique, "CabinetId must be unique"
no_demand = cabinets["demand"] <= 0
if no_demand.any():
    ids = cabinets.loc[no_demand, "CabinetId"].tolist()
    print(f"WARNING: skipping {len(ids)} cabinet(s) with zero demand: "
          + ", ".join(ids[:10]) + (" ..." if len(ids) > 10 else ""))
    cabinets = cabinets[~no_demand].reset_index(drop=True)

for cab, d in zip(cabinets["CabinetId"], cabinets["demand"]):
    assert d <= CAP, f"cabinet {cab} has demand {d:.3f} above CAP – this data needs a hotspot step"

TOTAL_DEMAND = cabinets["demand"].sum()
SMALLEST_MIN = MIN_PEOPLE * MIN_DEMAND_PER_PERSON   # a cluster below this is too small even for the smallest team
K_MIN = math.ceil(TOTAL_DEMAND / CAP - 1e-9)
K_MAX = max(K_MIN, math.floor(TOTAL_DEMAND / SMALLEST_MIN + 1e-9))
print(f"{len(cabinets)} cabinets, total demand = {TOTAL_DEMAND:.3f} tickets/day "
      f"(busiest cabinet {cabinets['demand'].max():.3f})")
print(f"Clusters: at least {K_MIN} (all full teams at {CAP}/day), at most {K_MAX} "
      f"(all teams of {MIN_PEOPLE} at {SMALLEST_MIN}/day)")
cabinets.describe().round(3)

## Distance helpers
* **Haversine distance** – straight-line distance in km between two lat/long points, following the earth's curve.
* **Local km projection** – k-means and the outlines need ordinary x/y coordinates. We flatten lat/long into
  kilometres around the data's mean latitude (an *equirectangular* projection). This is accurate for a region up to
  a few hundred km across (a city or a state); a much larger region would need a proper map projection.
* **Cluster centre** – the simple (unweighted) average of the member cabinets' latitude and longitude.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Great-circle distance in km; inputs broadcast like numpy arrays.
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


REF_LAT = cabinets["cabinetlat"].mean()
REF_LON = cabinets["cabinetlong"].mean()


def project_km(lat, lon):
    # Equirectangular projection to local km around the data's mean latitude.
    x = np.radians(np.asarray(lon) - REF_LON) * np.cos(np.radians(REF_LAT)) * EARTH_RADIUS_KM
    y = np.radians(np.asarray(lat) - REF_LAT) * EARTH_RADIUS_KM
    return np.column_stack([x, y])


def unproject_km(xy):
    # Inverse of project_km: local km (x, y) -> (long, lat) columns, the order map shapes use.
    lon = REF_LON + np.degrees(xy[:, 0] / (np.cos(np.radians(REF_LAT)) * EARTH_RADIUS_KM))
    lat = REF_LAT + np.degrees(xy[:, 1] / EARTH_RADIUS_KM)
    return np.column_stack([lon, lat])


# The cabinets as plain arrays, used by every step below (position i = row i of `cabinets`)
LAT = cabinets["cabinetlat"].to_numpy()
LON = cabinets["cabinetlong"].to_numpy()
DEM = cabinets["demand"].to_numpy()
XY = project_km(LAT, LON)


def members_of(labels, p):
    # Positions of the cabinets in cluster p.
    return np.where(labels == p)[0]


def centre_of(members, flagged=None):
    # Centre (lat, long) of these cabinets: the simple average of their positions. Cabinets marked in `flagged`
    # (leftovers, and in the road version straight-line fallbacks) don't move a centre; if every cabinet is flagged,
    # all of them are used.
    members = np.asarray(members, dtype=int)
    if flagged is not None and (~flagged[members]).any():
        members = members[~flagged[members]]
    return LAT[members].mean(), LON[members].mean()


def cluster_stats(labels, flagged=None):
    # For every cluster: (cluster labels, each cabinet's index into them, demand, centre lat, centre long).
    # The centre leaves out the cabinets marked in `flagged` (see centre_of).
    parts, inv = np.unique(labels, return_inverse=True)
    w = np.ones(len(labels)) if flagged is None else (~flagged).astype(float)
    w = np.where(np.bincount(inv, w)[inv] > 0, w, 1.0)      # a cluster with only flagged cabinets uses them all
    n = np.bincount(inv, w)
    return parts, inv, np.bincount(inv, DEM), np.bincount(inv, LAT * w) / n, np.bincount(inv, LON * w) / n


def dist_to_centre(labels, flagged=None):
    # Straight-line km from every cabinet to its own cluster centre.
    parts, inv, load, clat, clon = cluster_stats(labels, flagged)
    return haversine_km(LAT, LON, clat[inv], clon[inv])


def fits_radius(members, radius_km, skip=None):
    # True if every cabinet in `members` is within radius_km of the members' centre. Cabinets marked in `skip`
    # (leftovers) have no distance limit and don't move the centre.
    members = np.asarray(members, dtype=int)
    if len(members) == 0:
        return True
    clat, clon = centre_of(members, skip)
    check = members if skip is None else members[~skip[members]]
    return bool((haversine_km(LAT[check], LON[check], clat, clon) <= radius_km + 1e-9).all())

## Cluster outlines and neighbouring cabinets
**Outline.** A cluster's outline is the rubber band stretched around its cabinets (its *convex hull*: the smallest
shape with no dents that holds them all). One cabinet gives a point and two give a line. Two outlines **overlap** if
they share more than a border or a single point. Rule 3 says no two outlines may overlap.

**Why the rule can always be kept.** k-means gives every cabinet to its nearest centre, which cuts the map into
straight-edged slices, one per cluster. Each cluster's outline stays inside its own slice, so right after k-means no
outlines overlap. After that, every change is checked first and only made if the outlines stay clear:

* **Outline check** – before a cabinet moves (or two clusters merge, or a cluster is split), we draw the new outlines
  of the clusters that would change and make sure they don't overlap any other outline.

**Neighbouring cabinets** are only used to find *which* clusters are worth trying, and to shade the map:

* every spot on the map is given to its nearest cabinet; a cabinet's *area* is the part of the map closer to it than
  to any other cabinet (a *Voronoi diagram*). Areas stop 1.5 km beyond the outermost cabinets;
* two cabinets are **neighbours** when their areas share a border line; a cabinet **touches** a cluster when one of
  its neighbours is in that cluster. Moves are only tried into clusters a cabinet touches, which keeps the model fast
  with thousands of cabinets.

In [ ]:
AREA_MARGIN_KM = 1.5   # cabinet areas stop this far beyond the outermost cabinets
_OUTLINES = {}         # saved outlines, so unchanged clusters are not redrawn every time


def outline(members):
    # Rubber band (convex hull, in projected km) around the cabinets at these positions.
    members = np.sort(np.asarray(members, dtype=int))
    key = members.tobytes()
    if key not in _OUTLINES:
        if len(_OUTLINES) > 200_000:
            _OUTLINES.clear()
        _OUTLINES[key] = shapely.MultiPoint(XY[members]).convex_hull
    return _OUTLINES[key]


def overlap(a, b):
    # Two outlines overlap when they share more than a border or a single point.
    return a.intersects(b) and not a.touches(b)


def outlines_clear(changes, labels, skip=None):
    # Would a change keep every outline clear of every other? `changes` = {cluster label: its cabinet positions
    # after the change} for every cluster the change touches (an empty list for a cluster that disappears).
    # Cabinets marked in `skip` (leftovers) are left out of the outlines. Only clusters whose bounding box comes
    # near a new outline can overlap it, so only those outlines are drawn.
    new = []
    for m in changes.values():
        m = np.asarray(m, dtype=int)
        if skip is not None:
            m = m[~skip[m]]
        if len(m):
            new.append(outline(m))
    for k, shape in enumerate(new):
        if any(overlap(shape, other) for other in new[k + 1:]):
            return False
    keep = ~np.isin(labels, list(changes))
    if skip is not None:
        keep &= ~skip
    idx = np.where(keep)[0]
    if len(idx) == 0 or not new:
        return True
    order = idx[np.argsort(labels[idx], kind="stable")]       # the other clusters' cabinets, grouped by cluster
    starts = np.r_[0, np.flatnonzero(np.diff(labels[order])) + 1]
    ends = np.r_[starts[1:], len(order)]
    x, y = XY[order, 0], XY[order, 1]
    x0, x1 = np.minimum.reduceat(x, starts), np.maximum.reduceat(x, starts)
    y0, y1 = np.minimum.reduceat(y, starts), np.maximum.reduceat(y, starts)
    for shape in new:
        bx0, by0, bx1, by1 = shape.bounds
        for g in np.where((x0 <= bx1) & (x1 >= bx0) & (y0 <= by1) & (y1 >= by0))[0]:
            if overlap(shape, outline(order[starts[g]:ends[g]])):
                return False
    return True


def change_ok(changes, labels, radius_km, skip=None):
    # A change is allowed if every cluster it touches keeps all its cabinets within radius_km of its new centre
    # and no outlines overlap.
    return (all(fits_radius(m, radius_km, skip) for m in changes.values())
            and outlines_clear(changes, labels, skip))


def can_move(i, dest, labels, radius_km=None):
    # Moving cabinet i into cluster dest keeps the outlines clear (and, if radius_km is given, the distance rule).
    src = labels[i]
    rest = members_of(labels, src)
    changes = {src: rest[rest != i], dest: np.append(members_of(labels, dest), i)}
    return outlines_clear(changes, labels) if radius_km is None else change_ok(changes, labels, radius_km)


def cabinet_areas(margin_km):
    # Each cabinet's area (its Voronoi cell, in projected km), cut off margin_km beyond the outermost cabinets.
    border = shapely.MultiPoint(XY).convex_hull.buffer(margin_km)
    if len(XY) == 1:
        return [border]
    cells = [None] * len(XY)
    tree = shapely.STRtree(shapely.points(XY))
    for cell in shapely.voronoi_polygons(shapely.MultiPoint(XY), extend_to=border).geoms:
        for i in tree.query(cell, predicate="contains"):   # cabinets at the very same spot share one area
            cells[i] = cell.intersection(border)
    return cells


def neighbours_of(cells):
    # Two cabinets are neighbours when their areas share a border line (touching at one corner is not enough).
    tree = shapely.STRtree(cells)
    nbrs = [set() for _ in cells]
    for i, cell in enumerate(cells):
        for j in tree.query(cell, predicate="intersects"):
            if j != i and cell.intersection(cells[j]).length > 1e-9:
                nbrs[i].add(int(j))
    return nbrs


CABINET_CELLS = cabinet_areas(AREA_MARGIN_KM)
NEIGHBOURS = neighbours_of(CABINET_CELLS)
EDGES = np.array([(i, j) for i, js in enumerate(NEIGHBOURS) for j in js], dtype=int).reshape(-1, 2)


def touching_clusters(members, labels):
    # Clusters (other than the cabinets' own) that these cabinets touch.
    members = np.atleast_1d(members)
    return {labels[j] for i in members for j in NEIGHBOURS[i]} - set(labels[members])


def cluster_areas(cluster_of, skip=None):
    # One shape per cluster (in long/lat): the areas of its cabinets joined together (used for the map). Cabinets
    # marked in `skip` (flagged ones) are left out, so a cluster made only of them gets no shape.
    keep = np.ones(len(cluster_of), dtype=bool) if skip is None else ~skip
    return {c: shapely.transform(shapely.unary_union([CABINET_CELLS[i] for i in np.where((cluster_of == c) & keep)[0]]),
                                 unproject_km)
            for c in np.unique(cluster_of[keep])}


print(f"{len(cabinets)} cabinet areas, {len(EDGES) // 2} neighbour links "
      f"(on average {len(EDGES) / len(cabinets):.1f} neighbours per cabinet)")

## Demand Layer – split by demand
We decide how many clusters are needed and share the cabinets out between them, over the whole dataset at once.
The idea: "decide how many clusters are needed first, then share the work evenly".

1. `k = ceil(total demand / CAP)` – the fewest clusters that could possibly hold all the demand.
2. Run **demand-weighted k-means** with `k` groups on projected km coordinates. (k-means places `k` centre points and
   gives every cabinet to its nearest centre; *demand-weighted* means busy cabinets pull the centres towards them
   more strongly, which tends to even out the demand.) The outlines don't overlap at this point (see above).
3. **Repair loop** – k-means does not know about `CAP`, so a cluster may be slightly overloaded. While any cluster
   exceeds `CAP`: among the cabinets in the most overloaded cluster, take the one closest to the centre of a touching
   cluster that can take it without exceeding `CAP`, and move it to the **lightest** such cluster within `RADIUS_KM`
   of it (or the nearest one if none is that close) – as long as the outlines stay clear. If no touching cluster has
   room for a whole cabinet, we **pass one on** to a touching cluster that has a little room, as long as that lowers
   the total overload: the overload then moves on from that cluster to its own neighbours, like water finding a way
   out. Each move strictly reduces the total overload.
4. If the loop gets stuck (no move left, or 500 moves), the **most overloaded cluster is split in two** with
   k-means (k = 2) and the repair loop carries on – the two halves have room, so overload from the clusters around
   them can now flow into them. A split cuts a cluster along a straight line, so the outlines stay clear. Splitting
   one cluster at a time keeps the rest of the map as it is; halves that end up below `MIN_DEMAND` are dealt with in
   the minimum demand step.
5. **Balancing loop** – steps 2–3 only guarantee every cluster is ≤ `CAP`; they do not make the clusters *equal*.
   So we keep moving single cabinets from a heavier cluster to a lighter touching one, as long as the move brings
   the two loads closer together (cabinet demand < the gap between them). Cabinets on the **border** go first (the
   smallest extra distance), and a move is only allowed if the outlines stay clear, both clusters keep their cabinets
   within `RADIUS_KM` of their centres, and the receiving cluster stays ≤ `CAP`. The loop stops once the heaviest and
   lightest clusters are within `BALANCE_TOL` tickets/day of each other (or no such move remains). Because cabinets
   can't be split, an exactly equal split is not always possible anyway.

In [ ]:
MAX_REPAIR_MOVES = 500
MAX_BALANCE_MOVES = max(500, len(cabinets))


def repair_overloads(labels, cap, radius_km, max_moves=MAX_REPAIR_MOVES):
    # Move cabinets out of overloaded clusters into touching clusters. Returns (labels, n_moves, success).
    labels = labels.copy()
    for move in range(max_moves + 1):
        parts, inv, load_arr, clat, clon = cluster_stats(labels)
        load = dict(zip(parts, load_arr))
        over = [p for p in parts if load[p] > cap + 1e-9]
        if not over:
            return labels, move, True
        if move == max_moves:
            break
        src = max(over, key=lambda p: load[p])
        centre = {p: (la, lo) for p, la, lo in zip(parts, clat, clon)}

        def km(i, q):
            return haversine_km(LAT[i], LON[i], *centre[q])

        options = [(i, q) for i in members_of(labels, src) for q in touching_clusters(i, labels)]
        fit = [(i, q) for i, q in options if load[q] + DEM[i] <= cap + 1e-9]
        dest = None
        # cabinets in order of how close they are to a cluster that has room for them
        for i in sorted({i for i, _ in fit}, key=lambda i: min(km(i, q) for j, q in fit if j == i)):
            targets = [q for j, q in fit if j == i]
            close = sorted((q for q in targets if km(i, q) <= radius_km), key=lambda q: load[q])
            far = sorted((q for q in targets if km(i, q) > radius_km), key=lambda q: km(i, q))
            dest = next((q for q in close + far if can_move(i, q, labels)), None)
            if dest is not None:
                labels[i] = dest
                break
        if dest is not None:
            continue
        # No touching cluster can take a whole cabinet: pass one on to a touching cluster with a little room
        # anyway, if that lowers the total overload (the overload then moves on from there).
        passes = []
        for i, q in options:
            before = (load[src] - cap) + max(0, load[q] - cap)
            after = max(0, load[src] - DEM[i] - cap) + max(0, load[q] + DEM[i] - cap)
            if after < before - 1e-9:
                passes.append((after - before, km(i, q), i, q))
        choice = next(((i, q) for _, _, i, q in sorted(passes) if can_move(i, q, labels)), None)
        if choice is None:
            break   # nowhere to put anything: caller will increase k
        labels[choice[0]] = choice[1]
    return labels, move, False


def load_spread(labels):
    # Heaviest minus lightest cluster demand.
    load = cluster_stats(labels)[2]
    return load.max() - load.min()


def max_radius(members):
    # Largest straight-line distance (km) from one of these cabinets to their centre.
    return haversine_km(LAT[members], LON[members], LAT[members].mean(), LON[members].mean()).max()


def balance_loads(labels, cap, radius_km, tol, max_moves=MAX_BALANCE_MOVES):
    # Even out cluster demand with single-cabinet moves from heavier to lighter touching clusters until
    # heaviest - lightest <= tol. Each accepted move has dem[i] < load[src] - load[dst], which strictly
    # lowers the sum of squared deviations from the mean demand, so the loop always ends.
    # Returns (labels, n_moves).
    labels = labels.copy()
    for move in range(max_moves):
        parts, inv, load, clat, clon = cluster_stats(labels)
        if load.max() - load.min() <= tol:
            return labels, move
        # every (cabinet, touching cluster) pair, from the neighbour links (each pair coded as one number)
        code = np.unique(EDGES[:, 0] * len(parts) + inv[EDGES[:, 1]])
        i, q = code // len(parts), code % len(parts)
        own = inv[i]
        keep = (q != own) & (DEM[i] < load[own] - load[q] - 1e-9) & (load[q] + DEM[i] <= cap + 1e-9)
        i, q, own = i[keep], q[keep], own[keep]
        extra = (haversine_km(LAT[i], LON[i], clat[q], clon[q])          # extra km vs. staying
                 - haversine_km(LAT[i], LON[i], clat[own], clon[own]))
        moved = False
        for k in np.argsort(extra, kind="stable"):
            ci, src, dst = i[k], parts[own[k]], parts[q[k]]
            before = members_of(labels, src)
            rest, after = before[before != ci], np.append(members_of(labels, dst), ci)
            if len(rest) == 0 or max_radius(after) > radius_km:
                continue
            if max_radius(rest) > max(radius_km, max_radius(before)):
                continue
            if not outlines_clear({src: rest, dst: after}, labels):
                continue
            labels[ci] = dst
            moved = True
            break
        if not moved:
            return labels, move
    return labels, max_moves


def demand_layer(cap, radius_km, seed, balance_tol):
    # Returns (labels, summary): a cluster label for every cabinet, and what the repair and balancing loops did.
    k = min(math.ceil(DEM.sum() / cap - 1e-9), len(DEM))
    labels = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(XY, sample_weight=DEM)
    labels, moves, ok = repair_overloads(labels, cap, radius_km)
    splits = 0
    while not ok:
        # the repair loop is stuck: split the most overloaded cluster in two (its halves have room for the
        # neighbours' overload too) and carry on repairing
        parts, inv, load, clat, clon = cluster_stats(labels)
        members = members_of(labels, parts[np.argmax(load)])
        halves = KMeans(n_clusters=2, n_init=10, random_state=seed).fit_predict(XY[members])
        labels[members[halves == 1]] = labels.max() + 1
        splits += 1
        labels, more, ok = repair_overloads(labels, cap, radius_km)
        moves += more
    summary = {"k_start": k, "repair_moves": moves, "overloaded_clusters_split": splits,
               "clusters_after_repair": len(np.unique(labels)), "spread_before": load_spread(labels)}
    labels, summary["balance_moves"] = balance_loads(labels, cap, radius_km, balance_tol)
    summary["spread_after"] = load_spread(labels)
    return labels, pd.Series(summary, name="value").to_frame()

In [ ]:
demand_labels, demand_log = demand_layer(CAP, RADIUS_KM, SEED, BALANCE_TOL)
demand_loads = cluster_stats(demand_labels)[2]
print(f"Demand Layer: {len(demand_loads)} clusters (possible range {K_MIN} to {K_MAX}), "
      f"demand per cluster {demand_loads.min():.2f} to {demand_loads.max():.2f} tickets/day")
print(f"({time.time() - START:.0f} s so far) spread = heaviest minus lightest cluster (tickets/day), "
      "before and after balancing")
demand_log.round(2)

## Distance Layer – `RADIUS_KM` from each cluster centre
Now we check rule 2: every cabinet must be within `RADIUS_KM` (straight line) of its cluster centre. We repeatedly
take the cabinet that is **farthest** from its centre (if it is beyond `RADIUS_KM`) and fix it:

1. **Move it** to a touching cluster that can take its demand without going over `CAP`, where every cabinet
   (including this one) is within `RADIUS_KM` of the new centre and the outlines stay clear. If several clusters
   qualify, the lightest one wins.
2. If no cluster qualifies but the rest of its cluster is fine without it (every other cabinet within `RADIUS_KM`
   of the new centre, outlines clear), **take it out on its own**. This stops one far-off cabinet from breaking up
   its whole neighbourhood. The minimum demand step later finds it a home.
3. Otherwise **split** its cluster in two with k-means (k = 2). A split cuts the cluster along a straight line, so
   the two halves never overlap each other or anything else. Splitting never increases demand, so `CAP` still holds.

After every change the centres are recomputed and every cabinet is checked again. A cabinet is never moved into the
same cluster twice, so it can't bounce back and forth. The loop is capped and stops with an error if it doesn't
settle.

In [ ]:
MAX_DISTANCE_ROUNDS = max(300, len(cabinets))


def best_new_cluster(i, labels, cap, radius_km, tried):
    # The lightest touching cluster that can take cabinet i without going over cap, keeping every cabinet within
    # radius_km of its new centre and the outlines clear. None if no cluster qualifies.
    load = {p: DEM[labels == p].sum() for p in touching_clusters(i, labels)}
    options = sorted((q for q in load if (i, q) not in tried and load[q] + DEM[i] <= cap + 1e-9),
                     key=lambda q: load[q])
    return next((q for q in options if can_move(i, q, labels, radius_km)), None)


def distance_layer(labels, cap, radius_km, seed, max_rounds=MAX_DISTANCE_ROUNDS):
    # Fix every cabinet that is too far from its cluster centre, farthest first: move it, take it out on its own,
    # or split its cluster in two. Returns (labels, action log, number of cabinets too far at first).
    labels = labels.copy()
    tried, actions, first_fail = set(), [], None
    for rnd in range(max_rounds + 1):
        dist = dist_to_centre(labels)
        fail = np.where(dist > radius_km + 1e-9)[0]
        if first_fail is None:
            first_fail = len(fail)
        if len(fail) == 0:
            return labels, pd.DataFrame(actions), first_fail
        if rnd == max_rounds:
            break
        i = fail[np.argmax(dist[fail])]
        entry = {"round": rnd + 1, "CabinetId": cabinets["CabinetId"].iloc[i], "km_to_centre": dist[i]}
        dest = best_new_cluster(i, labels, cap, radius_km, tried)
        if dest is not None:
            labels[i] = dest
            tried.add((i, dest))
            entry["action"] = "moved to another cluster"
        else:
            members = members_of(labels, labels[i])
            rest, new = members[members != i], labels.max() + 1
            if change_ok({labels[i]: rest, new: [i]}, labels, radius_km):
                labels[i] = new
                entry["action"] = "taken out on its own"
            else:
                halves = KMeans(n_clusters=2, n_init=10, random_state=seed).fit_predict(XY[members])
                labels[members[halves == 1]] = new
                entry["action"] = "cluster split in two"
        actions.append(entry)
    raise RuntimeError(f"Distance Layer did not settle after {max_rounds} rounds: {len(fail)} cabinet(s) are "
                       f"still more than {radius_km} km from their cluster centre")

In [ ]:
labels, distance_log, first_fail = distance_layer(demand_labels, CAP, RADIUS_KM, SEED)
counts = distance_log["action"].value_counts() if len(distance_log) else pd.Series(dtype=int)
print(f"Distance check at first: {first_fail} of {len(cabinets)} cabinets were more than {RADIUS_KM} km "
      f"from their cluster centre")
print(f"Fixed with {counts.get('moved to another cluster', 0)} move(s), "
      f"{counts.get('taken out on its own', 0)} cabinet(s) taken out on their own and "
      f"{counts.get('cluster split in two', 0)} split(s): {len(np.unique(labels))} clusters now")
distance_log.round(2)

## Minimum demand – bring small clusters up to `MIN_DEMAND`
A cluster below `MIN_DEMAND` (12 tickets/day) is not a worthwhile day for a team. The Distance Layer and the
Demand Layer can leave such clusters behind. We take them one at a time, lightest first, and try in this order:

1. **Merge** it with a touching cluster, if together they stay ≤ `CAP`. We prefer a merge that reaches
   `MIN_DEMAND`, then the nearest cluster.
2. **Share it out**: hand its cabinets, one at a time, to touching clusters that have room. This only goes ahead if
   *every* cabinet finds a place; then the small cluster disappears.
3. **Top it up**: take over cabinets from touching clusters that stay at or above `MIN_DEMAND` after giving one
   away, nearest cabinet first, until it reaches `MIN_DEMAND`.

Every change must keep all clusters ≤ `CAP`, every cabinet within `RADIUS_KM` of its (new) centre, and the outlines
clear. A cluster where nothing works is put aside as *stuck for now*, and tried again whenever something else
changes – its neighbours may have changed in the meantime. Every successful change removes a cluster or brings a
small cluster closer to `MIN_DEMAND`, so the step always ends. A cluster that still can't reach `MIN_DEMAND` at the
end goes to the *Smaller teams* step below.

In [ ]:
MAX_MIN_ROUNDS = max(2000, 10 * len(cabinets))   # safety cap only; at 7.5 km Perak needs about 2 per cabinet


def min_demand_step(labels, cap, min_demand, radius_km, active=None, max_rounds=MAX_MIN_ROUNDS):
    # Bring every cluster below min_demand up to it, lightest first: merge, share out, top up.
    # `active` = the clusters this step may change (None = all of them).
    # Returns (labels, action log, labels of clusters left below min_demand).
    labels = labels.copy()
    active = None if active is None else {int(p) for p in active}

    def load(p):
        return DEM[labels == p].sum()

    def members(p):
        return members_of(labels, p)

    def km_to_centre(i, p):
        # straight-line km from cabinet i to the centre of cluster p
        m = members(p)
        return haversine_km(LAT[i], LON[i], LAT[m].mean(), LON[m].mean())

    def centre_gap(a, b):
        # straight-line km between the centres of clusters a and b
        ma, mb = members(a), members(b)
        return haversine_km(LAT[ma].mean(), LON[ma].mean(), LAT[mb].mean(), LON[mb].mean())

    def touching(m):
        # the clusters these cabinets touch that this step may change
        found = touching_clusters(m, labels)
        return found if active is None else found & active

    def merge(s):
        opts = sorted((t for t in touching(members(s)) if load(s) + load(t) <= cap + 1e-9),
                      key=lambda t: (load(s) + load(t) < min_demand - 1e-9, centre_gap(s, t)))
        for t in opts:
            if change_ok({s: [], t: np.concatenate([members(s), members(t)])}, labels, radius_km):
                labels[labels == s] = t
                return "merged with a neighbour"
        return None

    def share_out(s):
        before = labels.copy()
        while (labels == s).any():
            ms = members(s)
            opts = sorted(((i, t) for i in ms for t in touching(i)
                           if load(t) + DEM[i] <= cap + 1e-9),
                          key=lambda it: km_to_centre(*it))
            choice = next(((i, t) for i, t in opts
                           if change_ok({s: ms[ms != i], t: np.append(members(t), i)}, labels, radius_km)), None)
            if choice is None:
                labels[:] = before   # not every cabinet found a place: undo
                return None
            labels[choice[0]] = choice[1]
        return "shared out to neighbours"

    def top_up(s):
        taken = 0
        while load(s) < min_demand - 1e-9:
            ms = members(s)
            opts = sorted(((i, t) for t in touching(ms) for i in members(t)
                           if load(t) - DEM[i] >= min_demand - 1e-9 and load(s) + DEM[i] <= cap + 1e-9
                           and s in touching_clusters(i, labels)),
                          key=lambda it: km_to_centre(it[0], s))
            choice = next(((i, t) for i, t in opts
                           if change_ok({s: np.append(ms, i), t: members(t)[members(t) != i]}, labels, radius_km)),
                          None)
            if choice is None:
                break
            labels[choice[0]] = s
            taken += 1
        return f"took over {taken} cabinet(s)" if taken else None

    log, stuck = [], set()
    for _ in range(max_rounds):
        parts, inv, load_arr, clat, clon = cluster_stats(labels)
        small = [p for p, d in zip(parts, load_arr) if d < min_demand - 1e-9 and p not in stuck
                 and (active is None or p in active)]
        if not small:
            return labels, pd.DataFrame(log), sorted(stuck)
        s = min(small, key=load)
        ms = members(s)
        entry = {"n_cabinets": len(ms), "demand_before": load(s),
                 "cabinets": ", ".join(cabinets["CabinetId"].iloc[ms[:3]]) + (" ..." if len(ms) > 3 else "")}
        action = merge(s) or share_out(s) or top_up(s)
        if action is None:
            stuck.add(s)
            action = "stuck for now"
        else:
            stuck.clear()   # something changed: clusters that were stuck get another try
        entry["action"] = action
        log.append(entry)
    raise RuntimeError(f"Minimum demand step did not settle after {max_rounds} rounds")

In [ ]:
labels, min_log, still_low = min_demand_step(labels, CAP, MIN_DEMAND, RADIUS_KM)
kinds = min_log["action"].str.split(" ").str[0].value_counts() if len(min_log) else pd.Series(dtype=int)
print(f"{len(min_log)} small-cluster step(s): {kinds.get('merged', 0)} merged, {kinds.get('shared', 0)} shared out, "
      f"{kinds.get('took', 0)} topped up, {kinds.get('stuck', 0)} 'stuck for now'; "
      f"{len(np.unique(labels))} clusters now ({time.time() - START:.0f} s so far)")
if still_low:
    print(f"{len(still_low)} cluster(s) could not reach {MIN_DEMAND}/day: they are tried with a smaller team below")
min_log.round(2)

## Smaller teams – one person fewer each round
A cluster that could not reach `MIN_DEMAND` sits in a quiet area: there is not enough demand within `RADIUS_KM` for a
full team. Instead of breaking it up, we give it a **smaller team**. Every team size uses the same two numbers per
person: at least `MIN_DEMAND_PER_PERSON` (4) and at most `DEMAND_PER_PERSON` (6) tickets/day.

| Team | Minimum | Maximum |
|---|---|---|
| 3 people | 12 | 18 |
| 2 people | 8 | 12 |
| 1 person | 4 | 6 (special case: up to `SOLO_MAX_DEMAND`, 7) |

Clusters that reached `MIN_DEMAND` keep the full team and are not touched again. The others form the **pool**. We go
down one person at a time, from `PEOPLE_PER_CLUSTER - 1` to `MIN_PEOPLE`. In each round:

1. The pool gets one more **minimum demand** pass with the smaller team's numbers (merge, share out, top up), but
   only among the pool's own clusters.
2. Every pool cluster whose demand now fits the round's team gets that team and leaves the pool.

After the last round a pool cluster can be one of three things:

* **Between 6 and 7 tickets/day** – a little too much for one person. It tries to pass cabinets on to touching
  clusters until it is at 6 or less. If it can't, one person takes it as the **special case** (up to 7), marked
  `one person, special case` in the outputs and with a dashed ring on the map.
* **Between two bands** – for example 7.5 tickets/day: too much for one person, too little for two. It first passes
  cabinets on until it is at 7 or less (one person), otherwise takes cabinets over from touching clusters until it
  reaches 8 (two people). A neighbour only gives a cabinet away if its own team stays the same, and only takes one
  if its new demand still fits a team. If neither works, the cluster is broken up in the *Leftovers* step below.
* **Too small even for the smallest team** (below 4 tickets/day). It is broken up in the *Leftovers* step below.

At the end every cluster's team is the one its demand calls for, so there is no team of 3 under 12, no team of 2
under 8 and no team of 1 under 4. All three rules still hold: no change is made if it would push a cluster over its
team's maximum, put a cabinet more than `RADIUS_KM` from its centre, or make two outlines overlap.

In [ ]:
def team_band(people):
    # (minimum, maximum) tickets/day for a team of this many people.
    return people * MIN_DEMAND_PER_PERSON, people * DEMAND_PER_PERSON


def team_for(demand):
    # (team size, special case?) for this demand: the smallest team whose band holds it. If none does, one person
    # may still take up to SOLO_MAX_DEMAND as the special case. (None, False) if no team fits: below the smallest
    # minimum, or in a gap between two bands (7 to 8 tickets/day: too much for one person, too little for two).
    for team in range(MIN_PEOPLE, PEOPLE_PER_CLUSTER + 1):
        lo, hi = team_band(team)
        if lo - 1e-9 <= demand <= hi + 1e-9:
            return team, False
    if MIN_PEOPLE == 1 and team_band(1)[0] - 1e-9 <= demand <= SOLO_MAX_DEMAND + 1e-9:
        return 1, True
    return None, False


def fits_band(demand):
    # Some team fits this demand without the one-person special case.
    team, special = team_for(demand)
    return team is not None and not special


def smallest_team(demand):
    # The team for this demand. A demand no team fits (only a cluster of leftovers that no cluster had room for)
    # gets the fewest people who can carry it.
    team = team_for(demand)[0]
    return team if team is not None else min(PEOPLE_PER_CLUSTER,
                                             max(MIN_PEOPLE, math.ceil(demand / DEMAND_PER_PERSON - 1e-9)))


def bands_around(demand):
    # For a demand in a gap: (the most a team can take below it, (minimum, maximum) of the next band above it).
    teams = range(MIN_PEOPLE, PEOPLE_PER_CLUSTER + 1)
    below = [team_band(t)[1] for t in teams if team_band(t)[1] < demand]
    if MIN_PEOPLE == 1 and SOLO_MAX_DEMAND < demand:
        below.append(SOLO_MAX_DEMAND)
    above = [team_band(t) for t in teams if team_band(t)[0] > demand]
    return (max(below) if below else None), (min(above) if above else None)


def loads_of(labels):
    # {cluster label: demand}
    parts, inv, load, clat, clon = cluster_stats(labels)
    return dict(zip(parts.tolist(), load))


def km_to_cluster_centre(i, p, labels):
    # straight-line km from cabinet i to the centre of cluster p
    return haversine_km(LAT[i], LON[i], *centre_of(members_of(labels, p)))


def shed(labels, s, target, radius_km):
    # Pass cabinets of cluster s, one at a time, to touching clusters until s is at or below `target` tickets/day.
    # A cabinet only goes to a cluster whose new demand still fits a team (not the special case). Preferred:
    # clusters whose team stays the same, then a cabinet that gets s there at once, then the nearest centre.
    # All or nothing: returns the new labels, or None if s can't get there.
    labels = labels.copy()
    while True:
        load = loads_of(labels)
        if load[s] <= target + 1e-9:
            return labels
        ms = members_of(labels, s)
        opts = sorted((team_for(load[t] + DEM[i])[0] != team_for(load[t])[0], load[s] - DEM[i] > target + 1e-9,
                       km_to_cluster_centre(i, t, labels), i, t)
                      for i in ms if load[s] - DEM[i] >= SMALLEST_MIN - 1e-9
                      for t in touching_clusters(i, labels) if fits_band(load[t] + DEM[i]))
        choice = next(((i, t) for *_, i, t in opts
                       if change_ok({s: ms[ms != i], t: np.append(members_of(labels, t), i)}, labels, radius_km)),
                      None)
        if choice is None:
            return None
        labels[choice[0]] = choice[1]


def top_up_to(labels, s, band, radius_km):
    # Take over cabinets from touching clusters until s reaches band = (minimum, maximum). A neighbour only gives a
    # cabinet away if its own team stays the same. All or nothing: returns the new labels, or None.
    labels = labels.copy()
    lo, hi = band
    while True:
        load = loads_of(labels)
        if load[s] >= lo - 1e-9:
            return labels
        ms = members_of(labels, s)
        opts = sorted((km_to_cluster_centre(i, s, labels), i, t)
                      for t in touching_clusters(ms, labels) for i in members_of(labels, t)
                      if load[s] + DEM[i] <= hi + 1e-9 and s in touching_clusters(i, labels)
                      and team_for(load[t] - DEM[i])[0] == team_for(load[t])[0])
        choice = next(((i, t) for _, i, t in opts
                       if change_ok({s: np.append(ms, i), t: members_of(labels, t)[members_of(labels, t) != i]},
                                    labels, radius_km)), None)
        if choice is None:
            return None
        labels[choice[0]] = s


def smaller_teams_step(labels, still_low, radius_km):
    # Give every cluster a team size. Clusters that reached MIN_DEMAND keep the full team; the others (the pool)
    # are tried with one person fewer each round. Clusters left between two bands then pass cabinets on or take
    # some over. Returns (labels, team size per cluster label, labels of the clusters to break up, log).
    labels = labels.copy()
    pool = {int(p) for p in still_low}
    log = []

    def load(p):
        return DEM[labels == p].sum()

    def note(p, result):
        m = members_of(labels, p)
        log.append({"n_cabinets": len(m), "demand": DEM[m].sum(),
                    "cabinets": ", ".join(cabinets["CabinetId"].iloc[m[:3]]) + (" ..." if len(m) > 3 else ""),
                    "result": result})

    for team in range(PEOPLE_PER_CLUSTER - 1, MIN_PEOPLE - 1, -1):
        if not pool:
            break
        lo, hi = team_band(team)
        labels, _, _ = min_demand_step(labels, hi, lo, radius_km, active=pool)
        pool &= {int(p) for p in np.unique(labels)}            # clusters merged away are gone
        for p in sorted(pool):
            if lo - 1e-9 <= load(p) <= hi + 1e-9:
                pool.discard(p)
                note(p, f"team of {team}")

    # what is left in the pool, lightest first: a gap (e.g. 7 to 8/day), the special case (6 to 7) or too small
    for p in sorted(pool, key=load):
        if not (labels == p).any():
            continue
        before = load(p)
        if team_for(before)[0] is None and before >= SMALLEST_MIN - 1e-9:   # in a gap: pass on, or take over
            below, above = bands_around(before)
            new = shed(labels, p, below, radius_km) if below is not None else None
            if new is None and above is not None:
                new = top_up_to(labels, p, above, radius_km)
            if new is not None:
                labels = new
        if team_for(load(p))[1]:                    # special case: try to get down to a normal one-person day
            new = shed(labels, p, team_band(1)[1], radius_km)
            if new is not None:
                labels = new
        team, special = team_for(load(p))
        changed = f" (was {before:.2f}/day)" if abs(load(p) - before) > 1e-9 else ""
        if team is None:
            note(p, "no team fits: broken up as leftovers" + changed)
        else:
            note(p, f"team of {team}" + (", special case" if special else "") + changed)

    # every cluster's team follows from its demand; clusters no team fits are broken up
    people, break_up = {}, []
    for p in np.unique(labels):
        team = team_for(load(p))[0]
        if team is None:
            break_up.append(int(p))
        else:
            people[int(p)] = team
    return labels, people, break_up, pd.DataFrame(log)


labels, people, break_up, team_log = smaller_teams_step(labels, still_low, RADIUS_KM)
sizes = pd.Series(people).value_counts()
print("Team sizes so far: " + ", ".join(f"{sizes.get(t, 0)} cluster(s) of {t}"
                                        for t in range(PEOPLE_PER_CLUSTER, MIN_PEOPLE - 1, -1))
      + f" ({time.time() - START:.0f} s so far)")
if break_up:
    print(f"{len(break_up)} cluster(s) fit no team (too small, or between two bands): "
          "their cabinets are handled as leftovers below")
team_log.round(2)

## Leftovers – nearest cluster border
A cluster that no team fits (too small, or still between two bands) is broken up: its cabinets are **leftovers**.
Each leftover cabinet joins the cluster with the **nearest border** – the straight-line distance from the cabinet to
the cluster's outline. Clusters it **touches** are tried first; if none of them can take it, the nearest border
anywhere is used. There is no distance limit for leftovers; they are flagged as `leftover` in the outputs.

**Room and team size.** A cluster can take a leftover only if its new demand still fits a team: 4–6, 8–12 or 12–18
tickets/day, never between 7 and 8 and never above `CAP` (18). One person up to 7 (the special case) is used only if
no cluster can take it otherwise. The team changes to the one the new demand calls for – for example 2 people at
11.5/day plus a 1/day cabinet becomes 3 people at 12.5/day.

**Leftovers stay apart.** A leftover counts for its cluster's demand and team, but it does **not** move the cluster
centre, is not part of the outline or territory, and is drawn on the map as a grey dot with a red ring. So a far-off
leftover can't pull the centre away from the cluster's own cabinets.

Leftovers are handled one at a time, the cabinet closest to a border first. Among the clusters that could take a
leftover, we prefer one where the outlines stay clear *with* the leftover included. If none qualifies, the nearest one
is used anyway. Leftovers are left out of the outline check.

If no cluster can take them at all, the cabinets that are left stay together as their own small cluster, all flagged
as leftovers and marked `below minimum`. So **every cabinet always ends up in a cluster**, and the manager can see the
ones that did not fit.

In [ ]:
def leftover_step(labels, people, break_up, cap, radius_km):
    # Break up the clusters in `break_up`, lightest first: each of their cabinets joins the cluster with the nearest
    # border whose new demand still fits a team, and that cluster's team changes if it has to.
    # Returns (labels, team size per cluster label, leftover flag per cabinet, log).
    labels, people = labels.copy(), dict(people)
    leftover = np.zeros(len(DEM), dtype=bool)
    todo = {int(p) for p in break_up}
    borders, log = {}, []

    def km_to_centre(i, p):
        # leftovers don't move a centre
        return haversine_km(LAT[i], LON[i], *centre_of(members_of(labels, p), leftover))

    def km_to_border(i, p):
        # straight-line km from cabinet i to the outline of cluster p. Leftovers are not part of an outline, so it
        # stays the same while leftovers join.
        p = int(p)
        if p not in borders:
            m = members_of(labels, p)
            borders[p] = outline(m[~leftover[m]] if (~leftover[m]).any() else m)
        return borders[p].distance(shapely.Point(XY[i]))

    def clean(i, t, s):
        # t can take leftover i with its own cabinets still within radius_km and the outlines clear
        after = np.append(members_of(labels, t), i)
        no_limit = leftover.copy()
        no_limit[i] = True                  # the leftover itself has no distance limit
        hidden = leftover.copy()
        hidden[labels == s] = True          # the cluster being broken up no longer counts ...
        hidden[i] = False                   # ... but the leftover counts in t's outline
        return fits_radius(after, radius_km, no_limit) and outlines_clear({t: after, s: []}, labels, hidden)

    def entry(i, **more):
        return {"CabinetId": cabinets["CabinetId"].iloc[i], "demand": DEM[i], **more}

    def special(d):
        return team_for(d)[1]

    while True:
        load = loads_of(labels)
        low = [p for p in todo if p in load and team_for(load[p])[0] is None]
        if not low:
            break
        s = min(low, key=load.get)
        while (labels == s).any():
            load = loads_of(labels)
            ms = members_of(labels, s)
            near = [(i, t) for i in ms for t in touching_clusters(i, labels)]
            anywhere = [(i, t) for i in ms for t in load if t != s]
            # only where the new demand still fits a team: touching clusters first, then anywhere; the one-person
            # special case (up to SOLO_MAX_DEMAND) only if no normal band is possible
            opts = []
            for pairs, fits in ((near, fits_band), (near, special), (anywhere, fits_band), (anywhere, special)):
                opts = [(i, t) for i, t in pairs if fits(load[t] + DEM[i]) and load[t] + DEM[i] <= cap + 1e-9]
                if opts:
                    break
            if not opts:   # no cluster has room: the rest stay together as their own small cluster
                todo.discard(s)
                for i in ms:
                    leftover[i] = True
                    log.append(entry(i, km_to_border=0.0, km_to_new_centre=km_to_centre(i, s), outlines_clear=False,
                                     others_pushed_over_limit="",
                                     note="no cluster has room: kept as own cluster"))
                break
            opts.sort(key=lambda it: km_to_border(*it))
            choice = next(((i, t) for i, t in opts if clean(i, t, s)), None)
            i, t = choice if choice is not None else opts[0]
            t, km = int(t), km_to_border(i, t)
            labels[i], leftover[i] = t, True
            pushed = []
            if choice is None:   # the nearest cluster had to be used: flag its cabinets pushed over the limit
                m = members_of(labels, t)
                centre = centre_of(m, leftover)
                m = m[~leftover[m]]
                far = m[haversine_km(LAT[m], LON[m], *centre) > radius_km]
                leftover[far] = True
                borders.pop(t, None)
                pushed = list(cabinets["CabinetId"].iloc[far])
            team, is_special = team_for(load[t] + DEM[i])
            notes = []
            if people.get(t) is not None and team != people[t]:
                notes.append(f"team changed from {people[t]} to {team}")
            if is_special:
                notes.append("one person, special case")
            people[t] = team
            log.append(entry(i, km_to_border=km, km_to_new_centre=km_to_centre(i, t),
                             outlines_clear=choice is not None, others_pushed_over_limit=", ".join(pushed),
                             note="; ".join(notes)))
    # every cluster's final team follows from its demand
    return labels, {p: smallest_team(d) for p, d in loads_of(labels).items()}, leftover, pd.DataFrame(log)


labels, people, leftover, leftover_log = leftover_step(labels, people, break_up, CAP, RADIUS_KM)
dist = dist_to_centre(labels, leftover)
rule = np.where(leftover, "leftover", "ok")
print(f"{int(leftover.sum())} cabinet(s) flagged as leftover; {len(np.unique(labels))} clusters "
      f"({time.time() - START:.0f} s so far)")
leftover_log.round(2)

## Cluster tables
We turn the final cluster labels into two tables: one row per cluster (IDs `C001`, `C002`, …) and one row per
cabinet. `people` is the cluster's team size and `demand_per_person` shows how the day's demand splits across that
team. `demand_rule` is `ok`, `one person, special case` (6 to 7 tickets/day for one person), or `below minimum` for a
cluster of leftovers that no cluster could take. The centre (`centre_lat`, `centre_long`) is the average position of
the cluster's own cabinets; leftovers are left out.

In [ ]:
def build_tables(labels, dist, rule, people):
    # Turn cluster labels into the clusters / assignments tables.
    cluster_rows, assign_rows = [], []
    for n, p in enumerate(np.unique(labels), start=1):
        cid = f"C{n:03d}"
        m = labels == p
        demand, team = DEM[m].sum(), people[int(p)]
        fits, special = team_for(demand)
        lo, hi = team_band(team)
        hi = SOLO_MAX_DEMAND if special else hi
        clat, clon = centre_of(np.where(m)[0], rule != "ok")    # flagged cabinets don't move the centre
        cluster_rows.append({"cluster_id": cid, "n_cabinets": int(m.sum()), "people": team,
                             "demand_per_day": demand, "min_demand": lo, "max_demand": hi,
                             "demand_rule": ("below minimum" if fits is None
                                             else "one person, special case" if special else "ok"),
                             "fill_rate": demand / hi, "demand_per_person": demand / team,
                             "max_dist_to_centre_km": dist[m].max(),
                             "centre_lat": clat, "centre_long": clon,
                             "n_leftover": int((rule[m] == "leftover").sum())})
        for i in np.where(m)[0]:
            assign_rows.append({"CabinetId": cabinets["CabinetId"].iloc[i], "cluster_id": cid, "people": team,
                                "allocated_demand": DEM[i], "dist_to_centre_km": dist[i],
                                "distance_rule": rule[i]})
    return pd.DataFrame(cluster_rows), pd.DataFrame(assign_rows)


clusters, assignments = build_tables(labels, dist, rule, people)
clusters.head(10).round(3)

## Validation
These are hard checks (`assert`). If any rule is broken the notebook stops with an error instead of producing
misleading outputs. Everything is recomputed from the assignments table, not taken from the steps above:

1. Total allocated demand equals total input demand (to within 1e-6), and so does every individual cabinet's.
2. Every cluster's team is the one its demand calls for: 12–18 → 3 people, 8–12 → 2, 4–6 → 1, 6–7 → 1 as the
   special case. No team of 3 under 12, of 2 under 8, or of 1 under 4. Only a cluster made entirely of leftovers may
   fit no team (`below minimum`).
3. Every cabinet is within `RADIUS_KM` of its cluster centre (straight line). The centre leaves out leftovers, and
   leftovers have no distance limit.
4. No empty clusters; every cabinet is in exactly one cluster.
5. No two cluster outlines overlap (every pair is checked; leftovers are left out of the outlines).

In [ ]:
def validate(cabinets, clusters, assignments, radius_km):
    # 1. demand conservation
    assert abs(assignments["allocated_demand"].sum() - cabinets["demand"].sum()) < TOL, \
        "allocated demand != input demand"
    per_cab = assignments.groupby("CabinetId")["allocated_demand"].sum()
    gap = per_cab.reindex(cabinets["CabinetId"]).fillna(0).values - cabinets["demand"].values
    assert np.abs(gap).max() < TOL, "some cabinet's demand is not fully allocated"
    # 2. team size and demand (the clusters table must agree): every team is the one its demand calls for
    load = assignments.groupby("cluster_id")["allocated_demand"].sum()
    info = clusters.set_index("cluster_id").reindex(load.index)
    team = info["people"]
    assert team.between(MIN_PEOPLE, PEOPLE_PER_CLUSTER).all(), "a cluster's team size is out of range"
    fit = [team_for(d) for d in load]
    assert (team == [smallest_team(d) for d in load]).all(), "a cluster's team is not the one its demand calls for"
    expected = ["below minimum" if t is None else "one person, special case" if sp else "ok" for t, sp in fit]
    assert (info["demand_rule"] == expected).all(), "demand_rule does not match the cluster's demand"
    low = info["demand_rule"] == "below minimum"
    special = info["demand_rule"] == "one person, special case"
    hi = np.where(special, SOLO_MAX_DEMAND, team * DEMAND_PER_PERSON)
    assert (load[~low] >= team[~low] * MIN_DEMAND_PER_PERSON - TOL).all(), "a cluster is below its team's minimum"
    assert (load[~low] <= hi[~low] + TOL).all(), "a cluster is above its team's maximum"
    assert (team[special] == 1).all(), "the special case is for one person only"
    # only a cluster made of leftovers that no cluster had room for may fit no team
    all_left = assignments.groupby("cluster_id")["distance_rule"].apply(lambda r: (r == "leftover").all())
    assert all_left.reindex(load.index)[low].all(), "a cluster fits no team but is not made of leftovers"
    assert (assignments["people"] == assignments["cluster_id"].map(team)).all(), "assignments table is stale"
    assert np.allclose(load.reindex(clusters["cluster_id"]), clusters["demand_per_day"]), "clusters table is stale"
    # 3. distance to the (recomputed) centre
    #    The centre leaves out leftovers (a cluster made only of leftovers uses them all).
    m = assignments.merge(cabinets[["CabinetId", "cabinetlat", "cabinetlong"]], on="CabinetId")
    assert m["distance_rule"].isin(["ok", "leftover"]).all(), "unknown distance_rule"
    normal = (m["distance_rule"] == "ok").to_numpy()
    every = m.groupby("cluster_id")[["cabinetlat", "cabinetlong"]].mean()
    centres = m[normal].groupby("cluster_id")[["cabinetlat", "cabinetlong"]].mean().reindex(every.index).fillna(every)
    table = clusters.set_index("cluster_id")[["centre_lat", "centre_long"]].reindex(centres.index)
    assert np.allclose(table.to_numpy(), centres.to_numpy()), "cluster centres must leave out leftovers"
    centre = centres.reindex(m["cluster_id"])
    km = haversine_km(m["cabinetlat"].to_numpy(), m["cabinetlong"].to_numpy(),
                      centre["cabinetlat"].to_numpy(), centre["cabinetlong"].to_numpy())
    too_far = m.loc[normal & (km > radius_km + 1e-9), "CabinetId"].tolist()
    assert not too_far, f"cabinet(s) more than RADIUS_KM from their cluster centre: {too_far}"
    # 4. structure
    assert (clusters["n_cabinets"] > 0).all() and (clusters["demand_per_day"] > 0).all(), "empty cluster"
    assert set(clusters["cluster_id"]) == set(assignments["cluster_id"]), "cluster/assignment mismatch"
    counts = assignments["CabinetId"].value_counts()
    assert counts.reindex(cabinets["CabinetId"]).eq(1).all() and len(counts) == len(cabinets), \
        "a cabinet is not in exactly one cluster"
    # 5. no overlapping outlines, recomputed from scratch for every pair (leftovers left out)
    xy = project_km(m["cabinetlat"], m["cabinetlong"])
    ids, shapes = [], []
    for cid, rows in m[normal].groupby("cluster_id").groups.items():
        ids.append(cid)
        shapes.append(shapely.MultiPoint(xy[m.index.get_indexer(rows)]).convex_hull)
    tree = shapely.STRtree(shapes)
    a, b = tree.query(shapes, predicate="intersects")
    clash = sorted({tuple(sorted((ids[x], ids[y]))) for x, y in zip(a, b)
                    if x < y and not shapes[x].touches(shapes[y])})
    assert not clash, f"overlapping cluster outlines: {clash[:10]}"
    print(f"all checks passed: {len(clusters)} clusters ({int(low.sum())} below minimum, "
          f"{int(special.sum())} one-person special case), {len(assignments)} cabinets "
          f"({int((~normal).sum())} leftover), demand per cluster {load.min():.2f} to {load.max():.2f}, "
          f"max distance to centre {km[normal].max():.2f} km, no overlapping outlines")


validate(cabinets, clusters, assignments, RADIUS_KM)

## Secondary cabinets
Every cluster keeps its own cabinets – its **primary** cabinets. Around each cluster centre we also look at the
`RADIUS_KM` circle: any cabinet that belongs to *another* cluster but lies within `RADIUS_KM` of this centre (straight
line) is listed as one of this cluster's **secondary** cabinets. They are backup work close by that a team could pick
up, for example when it finishes early.

* A cabinet is primary in exactly one cluster, but can be secondary for several clusters.
* Secondaries are a list only: they are **not** counted in the cluster's demand, and the demand rule applies to
  primary cabinets only.
* Only the **nearest `MAX_SECONDARY` (10)** secondary cabinets per cluster are kept (nearest by straight line), so every cluster gets a short
  backup list. Set `MAX_SECONDARY = None` to keep them all.
* Results: `secondaries.csv` (one row per cluster–cabinet pair) and two new columns in `clusters.csv`:
  `n_secondary` and `secondary_demand_per_day`.

In [ ]:
def find_secondaries(clusters, assignments, radius_km, max_n):
    # The nearest max_n (cluster, cabinet) pairs per cluster where the cabinet belongs to another cluster but lies within radius_km
    # (straight line) of this cluster's centre.
    primary = assignments.set_index("CabinetId")["cluster_id"].reindex(cabinets["CabinetId"]).to_numpy()
    cids = clusters["cluster_id"].to_numpy()
    km = haversine_km(clusters["centre_lat"].to_numpy()[:, None], clusters["centre_long"].to_numpy()[:, None],
                      LAT[None, :], LON[None, :])                        # clusters x cabinets
    ci, i = np.nonzero(km <= radius_km + 1e-9)
    keep = cids[ci] != primary[i]
    ci, i = ci[keep], i[keep]
    found = (pd.DataFrame({"cluster_id": cids[ci], "CabinetId": cabinets["CabinetId"].to_numpy()[i],
                          "primary_cluster_id": primary[i], "demand": DEM[i], "dist_to_centre_km": km[ci, i]})
            .sort_values(["cluster_id", "dist_to_centre_km"]))
    if max_n is not None:
        found = found.groupby("cluster_id").head(max_n)   # keep only the nearest max_n per cluster
    return found.reset_index(drop=True)


secondaries = find_secondaries(clusters, assignments, RADIUS_KM, MAX_SECONDARY)
per_cluster = secondaries.groupby("cluster_id")
clusters["n_secondary"] = clusters["cluster_id"].map(per_cluster.size()).fillna(0).astype(int)
clusters["secondary_demand_per_day"] = clusters["cluster_id"].map(per_cluster["demand"].sum()).fillna(0.0)

# checks: a cabinet is never secondary for its own cluster, and every secondary is within the radius (recomputed)
assert (secondaries["cluster_id"] != secondaries["primary_cluster_id"]).all(), "a cabinet is secondary for its own cluster"
check = secondaries.merge(clusters[["cluster_id", "centre_lat", "centre_long"]], on="cluster_id") \
                   .merge(cabinets[["CabinetId", "cabinetlat", "cabinetlong"]], on="CabinetId")
km = haversine_km(check["cabinetlat"], check["cabinetlong"], check["centre_lat"], check["centre_long"])
assert (km <= RADIUS_KM + 1e-9).all(), "a secondary cabinet is outside RADIUS_KM"
print(f"{len(secondaries)} secondary links: {int((clusters['n_secondary'] > 0).sum())} of {len(clusters)} clusters have "
      f"secondary cabinets, on average {clusters['n_secondary'].mean():.1f} per cluster "
      f"({secondaries['CabinetId'].nunique()} different cabinets)")
clusters[["cluster_id", "n_cabinets", "demand_per_day", "n_secondary", "secondary_demand_per_day"]].head(10).round(2)

### Result summary
* **Clusters** vs the **possible range**: at least `ceil(total demand / CAP)` (every cluster full) and at most
  `floor(total demand / 4)` (every cluster a one-person day at its minimum). The closer to the fewest, the fewer days
  of work to plan.
* **Clusters by team size** – how many clusters got a team of 3, 2 or 1.
* **People needed per day** – the team sizes of all clusters added up. If every cluster had a full team it would be
  clusters × `PEOPLE_PER_CLUSTER`. The list above the table gives every cluster's team (`C001 : 3 People`); it is
  also saved as `teams.csv`.
* **Fill rate** = cluster demand / its team's maximum (7 for the one-person special case). It is at least 57% (4 out
  of 7) unless the cluster is marked `below minimum`.
* **Demand per person** = cluster demand / the cluster's team size; at most `DEMAND_PER_PERSON` (6).
* **One-person special case** – clusters of 6 to 7 tickets/day that one person takes because they couldn't get down
  to 6.
* **Clusters below their minimum** – clusters made of leftovers that no cluster could take.
* **Worst demand gap between clusters** – heaviest minus lightest cluster (tickets/day). Small means the work is
  shared out evenly.
* **Max distance to centre** – the most spread-out cluster, leaving out leftovers; always ≤ `RADIUS_KM`.
* **Leftover cabinets** – cabinets from clusters too small even for the smallest team; they joined the cluster with the
  nearest border.
* **Secondary cabinets** – links from a cluster to cabinets of other clusters within its radius (backup work; not counted in demand).

In [ ]:
fr = clusters["fill_rate"]
loads = clusters["demand_per_day"]
normal = assignments["distance_rule"] == "ok"
teams = clusters["people"].value_counts()
team_list = [f"{c} : {n} {'Person' if n == 1 else 'People'}" for c, n in zip(clusters["cluster_id"], clusters["people"])]
print(f"People needed per day: {int(clusters['people'].sum())}")
for k in range(0, len(team_list), 6):
    print("    ".join(f"{t:<15}" for t in team_list[k:k + 6]))
pd.Series({
    "Clusters": f"{len(clusters)}",
    "Possible range": f"{K_MIN} to {K_MAX}",
    "Clusters by team size": ", ".join(f"{teams.get(t, 0)} of {t}"
                                       for t in range(PEOPLE_PER_CLUSTER, MIN_PEOPLE - 1, -1)),
    "People needed per day": f"{int(clusters['people'].sum())} "
                             f"(all full teams: {len(clusters) * PEOPLE_PER_CLUSTER})",
    "Biggest team / tickets per person per day":
        f"{PEOPLE_PER_CLUSTER} / {MIN_DEMAND_PER_PERSON} to {DEMAND_PER_PERSON}",
    "Mean fill rate": f"{fr.mean():.1%}",
    "Min fill rate": f"{fr.min():.1%}",
    "Mean demand per person (tickets/day)": f"{clusters['demand_per_person'].mean():.2f}",
    "One-person special case (up to SOLO_MAX_DEMAND)":
        f"{int((clusters['demand_rule'] == 'one person, special case').sum())}",
    "Clusters below their minimum": f"{int((clusters['demand_rule'] == 'below minimum').sum())}",
    "Worst demand gap between clusters (tickets/day)": f"{loads.max() - loads.min():.2f}",
    "Max distance to centre, without leftovers (km)": f"{assignments.loc[normal, 'dist_to_centre_km'].max():.2f}",
    "Leftover cabinets": f"{int((~normal).sum())}",
    "Secondary cabinet links": f"{len(secondaries)}",
    "Mean secondary cabinets per cluster": f"{clusters['n_secondary'].mean():.1f}",
    "Run time (s)": f"{time.time() - START:.0f}",
}, name="value").to_frame()

## Save outputs
* `outputs/clusters.csv` – one row per cluster: size, team size (`people`), demand, the team's minimum and maximum,
  `demand_rule` (`ok`, `one person, special case` or `below minimum`), fill rate, demand per person, largest distance to the centre, centre
  position (leftovers left out), number of leftover cabinets.
* `outputs/teams.csv` – the team per cluster (`C001 : 3 People`, …) with a last row for the total people per day.
* `outputs/assignments.csv` – one row per cabinet: its cluster, the cluster's team size, its demand, its distance to the cluster centre, and
  `distance_rule` – `ok`, or `leftover` for a cabinet that joined the cluster with the nearest border without meeting the rules.
* `secondaries.csv` – one row per cluster and secondary cabinet: the cabinet, its own (primary) cluster, its demand and its distance to this cluster's centre.

In [ ]:
def team_table(clusters):
    # One row per cluster with its team ("C001 : 3 People"), and a last row with the total people per day.
    teams = clusters[["cluster_id", "people", "demand_per_day", "demand_rule"]].copy()
    teams.insert(1, "team", [f"{c} : {n} {'Person' if n == 1 else 'People'}"
                             for c, n in zip(teams["cluster_id"], teams["people"])])
    total = pd.DataFrame([{"cluster_id": "Total", "team": f"{int(teams['people'].sum())} People per day",
                           "people": int(teams["people"].sum()), "demand_per_day": teams["demand_per_day"].sum(),
                           "demand_rule": ""}])
    return pd.concat([teams, total], ignore_index=True)


def save_tables(clusters, assignments, secondaries):
    clusters.round(4).to_csv(OUT_DIR / "clusters.csv", index=False)
    team_table(clusters).round(4).to_csv(OUT_DIR / "teams.csv", index=False)
    assignments.round(4).to_csv(OUT_DIR / "assignments.csv", index=False)
    secondaries.round(4).to_csv(OUT_DIR / "secondaries.csv", index=False)


save_tables(clusters, assignments, secondaries)
print(f"saved clusters.csv ({len(clusters)} rows), teams.csv, assignments.csv ({len(assignments)} rows), "
      f"secondaries.csv ({len(secondaries)} rows)")

## Interactive map
`outputs/map.html` – open in any browser (also shown below). Layers can be switched on and off in the top-right corner.
The box at the bottom left shows the number of clusters and the total people needed per day.

* **Cluster outlines** – a solid line around each cluster's cabinets (the rubber band of rule 3, padded by 200 m so
  that one- and two-cabinet clusters show up). No two outlines overlap. A cluster with a smaller team has a **dashed** outline.
* **Territories** – the map shaded by cluster: every spot belongs to the cluster of its nearest cabinet, with a
  thin line where two clusters meet. Touching clusters get different colours (only 8 colours are used, so clusters
  far apart may share one). Leftovers are left out, so a small gap shows around them.
* **`RADIUS_KM` circles** – a dashed circle of `RADIUS_KM` around every centre (off at first; switch it on to see the
  distance rule). Every cabinet except leftovers lies inside its own cluster's circle.
* **Cluster centres** – a large round marker at each centre in the cluster's colour, with the team size written in
  it. A dashed white ring marks the one-person special case. Click it for the centre's position, demand and team.
* **Cluster names** – a label next to each centre, e.g. `C001 : 3 people` (switch off when zoomed out).
* **Cabinets** – a dot coloured by its cluster; dot size grows with demand. Click a cabinet to see its cluster, team,
  demand, its own lat/long, its cluster centre's lat/long and the km between them.
* **Leftover cabinets (drawn apart)** – grey dots with a red ring, in their own layer. They count for their cluster's
  demand and team, but are not drawn in its colour or territory and do not move its centre. The popup says which
  cluster they belong to.
* **Secondary links** – a thin dashed line from each centre to each of its secondary cabinets (off at first). Cabinet popups list the clusters a cabinet is secondary for; centre popups show how many secondary cabinets a cluster has.

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
FLAGGED_FILL = "#9aa0a6"   # flagged cabinets are grey, not in their cluster's colour


def colour_clusters(areas):
    # Greedy colouring: clusters whose territories touch get different colours (most neighbours first).
    ids, shapes = list(areas), list(areas.values())
    a, b = shapely.STRtree(shapes).query(shapes, predicate="intersects")
    neighbours = {c: [] for c in ids}
    for x, y in zip(a, b):
        if x != y:
            neighbours[ids[x]].append(ids[y])
    colours = {}
    for a in sorted(areas, key=lambda a: -len(neighbours[a])):
        taken = {colours[b] for b in neighbours[a] if b in colours}
        free = [c for c in PALETTE if c not in taken]
        colours[a] = free[0] if free else PALETTE[len(colours) % len(PALETTE)]
    return colours


def people_text(n):
    return f"{n} {'person' if n == 1 else 'people'}"


def centre_icon(team, colour, special):
    # A large round marker in the cluster's colour with the team size in it (dashed ring = one-person special case).
    ring = "dashed" if special else "solid"
    return folium.DivIcon(icon_size=(32, 32), icon_anchor=(16, 16),
                          html=f"<div style='width:24px;height:24px;border-radius:50%;background:{colour};"
                               f"border:4px {ring} #fff;box-shadow:0 0 0 2px #111,0 2px 8px rgba(0,0,0,.6);"
                               f"color:#fff;font:bold 15px/24px sans-serif;text-align:center;"
                               f"text-shadow:0 0 3px #000'>{team}</div>")


def name_icon(cid, team):
    # The cluster's name and team size, next to its centre.
    return folium.DivIcon(icon_size=(150, 20), icon_anchor=(-20, 10),
                          html=f"<div style='display:inline-block;white-space:nowrap;background:#fff;"
                               f"border:1px solid #111;border-radius:3px;padding:0 5px;"
                               f"font:bold 12px/18px sans-serif;color:#111'>{cid} : {people_text(team)}</div>")


def make_map(clusters, assignments, secondaries, radius_km, people, min_people):
    fmap = folium.Map(location=[REF_LAT, REF_LON], zoom_start=11, tiles="OpenStreetMap")
    by_cab = assignments.set_index("CabinetId").reindex(cabinets["CabinetId"])
    cluster_of = by_cab["cluster_id"].to_numpy()
    flagged = (by_cab["distance_rule"] != "ok").to_numpy()
    areas = cluster_areas(cluster_of, flagged)          # flagged cabinets are not part of a cluster's territory
    colours = colour_clusters(areas)
    for cid in clusters["cluster_id"]:
        colours.setdefault(cid, FLAGGED_FILL)           # a cluster made only of flagged cabinets
    territory_layer = folium.FeatureGroup(name="Territories")
    outline_layer = folium.FeatureGroup(name="Cluster outlines")
    circle_layer = folium.FeatureGroup(name=f"{radius_km:g} km circles", show=False)
    centre_layer = folium.FeatureGroup(name="Cluster centres")
    name_layer = folium.FeatureGroup(name="Cluster names")
    dot_layer = folium.FeatureGroup(name="Cabinets (coloured by cluster)")
    flag_layer = folium.FeatureGroup(name="Leftover cabinets (drawn apart)")
    info = clusters.set_index("cluster_id")

    for cid, shape in areas.items():
        p = info.loc[cid]
        folium.GeoJson({"type": "Feature", "geometry": mapping(shape), "properties": {}},
                       style_function=lambda _, c=colours[cid]: {"fillColor": c, "fillOpacity": 0.15,
                                                                 "color": "#555555", "weight": 0.8},
                       tooltip=f"{cid}: team of {p['people']}, {p['demand_per_day']:.2f}/day "
                               f"({p['fill_rate']:.0%} full), "
                               f"{p['n_cabinets']} cabinets").add_to(territory_layer)

    for cid, p in info.iterrows():
        special = p["demand_rule"] == "one person, special case"
        members = np.where((cluster_of == cid) & ~flagged)[0]
        if len(members):
            band = shapely.transform(outline(members).buffer(0.2), unproject_km)
            dash = None if p["people"] == people else "8 6"    # dashed outline for a smaller team
            folium.GeoJson({"type": "Feature", "geometry": mapping(band), "properties": {}},
                           style_function=lambda _, c=colours[cid], d=dash: {"fillColor": c, "fillOpacity": 0.1,
                                                                             "color": c, "weight": 2.5,
                                                                             "dashArray": d},
                           tooltip=f"outline of {cid} (team of {p['people']})").add_to(outline_layer)
        folium.Circle([p["centre_lat"], p["centre_long"]], radius=radius_km * 1000, color=colours[cid],
                      weight=1, dash_array="6 6", fill=False).add_to(circle_layer)
        popup = (f"<b>Centre of {cid}</b><br><b>{people_text(p['people'])}</b><br>"
                 f"centre lat, long: {p['centre_lat']:.5f}, {p['centre_long']:.5f}<br>"
                 f"{p['n_cabinets']} cabinets<br>"
                 f"demand: {p['demand_per_day']:.2f}/day ({p['fill_rate']:.0%} full), "
                 f"{p['demand_per_person']:.2f} per person<br>"
                 f"farthest cabinet: {p['max_dist_to_centre_km']:.2f} km<br>"
                 f"team of {p['people']}: {p['min_demand']:g} to {p['max_demand']:g} tickets/day")
        if special:
            popup += f"<br><b>special case</b>: one person, up to {SOLO_MAX_DEMAND:g} tickets/day"
        elif p["demand_rule"] == "below minimum":
            popup += "<br><b>below minimum</b>: leftovers that no cluster had room for"
        if p["n_leftover"]:
            popup += f"<br>leftover cabinets (drawn apart, not used for the centre): {p['n_leftover']}"
        popup += (f"<br>secondary cabinets: {p['n_secondary']} "
                  f"({p['secondary_demand_per_day']:.2f} tickets/day)")
        folium.Marker([p["centre_lat"], p["centre_long"]], icon=centre_icon(p["people"], colours[cid], special),
                      tooltip=f"{cid} : {people_text(p['people'])}", z_index_offset=1000,
                      popup=folium.Popup(popup, max_width=280)).add_to(centre_layer)
        folium.Marker([p["centre_lat"], p["centre_long"]], icon=name_icon(cid, p["people"]),
                      interactive=False).add_to(name_layer)

    position = {cab: i for i, cab in enumerate(cabinets["CabinetId"])}
    secondary_for = secondaries.groupby("CabinetId")["cluster_id"].apply(lambda s: ", ".join(sorted(s)))
    for a in assignments.itertuples():
        i = position[a.CabinetId]
        p = info.loc[a.cluster_id]
        left = a.distance_rule == "leftover"
        popup = (f"<b>{a.CabinetId}</b><br>cluster: {a.cluster_id} ({people_text(a.people)})<br>"
                 f"demand: {a.allocated_demand:.3f}/day<br>"
                 f"cabinet lat, long: {LAT[i]:.5f}, {LON[i]:.5f}<br>"
                 f"centre of {a.cluster_id}: {p['centre_lat']:.5f}, {p['centre_long']:.5f}<br>"
                 f"distance to cluster centre: {a.dist_to_centre_km:.2f} km")
        if left:
            popup += (f"<br><b>leftover</b>: its own cluster was too small for a team, so it joined the nearest "
                      f"cluster border. Its demand counts for {a.cluster_id}, but it is drawn apart and does not "
                      f"move the centre.")
        if a.CabinetId in secondary_for.index:
            popup += f"<br>secondary for: {secondary_for[a.CabinetId]}"
        folium.CircleMarker([LAT[i], LON[i]], radius=3 + 2.5 * np.sqrt(DEM[i]),
                            color="#d00000" if left else "#ffffff", weight=3 if left else 1, fill=True,
                            fill_color=FLAGGED_FILL if left else colours[a.cluster_id], fill_opacity=0.9,
                            tooltip=f"{a.CabinetId}" + (" (leftover)" if left else ""),
                            popup=folium.Popup(popup, max_width=280)).add_to(flag_layer if left else dot_layer)

    link_layer = folium.FeatureGroup(name="Secondary links", show=False)
    for s in secondaries.itertuples():
        p, i = info.loc[s.cluster_id], position[s.CabinetId]
        folium.PolyLine([[p["centre_lat"], p["centre_long"]], [LAT[i], LON[i]]], color=colours[s.cluster_id],
                        weight=1, opacity=0.7, dash_array="4 4",
                        tooltip=f"{s.CabinetId}: secondary for {s.cluster_id}").add_to(link_layer)

    for layer in (territory_layer, outline_layer, circle_layer, link_layer, dot_layer, flag_layer, name_layer,
                  centre_layer):
        layer.add_to(fmap)
    folium.LayerControl(collapsed=False).add_to(fmap)
    title = (f"<div style='position:fixed;bottom:24px;left:10px;z-index:9999;background:white;padding:6px 10px;"
             f"border-radius:4px;font:14px sans-serif;box-shadow:0 1px 4px rgba(0,0,0,.3)'>"
             f"<b>{len(clusters)} clusters, {int(clusters['people'].sum())} people per day</b><br>"
             f"teams of {min_people} to {people} (number on the centre = team size), "
             f"{radius_km:g} km radius ({2 * radius_km:g} km across)</div>")
    fmap.get_root().html.add_child(folium.Element(title))
    return fmap


fmap = make_map(clusters, assignments, secondaries, RADIUS_KM, PEOPLE_PER_CLUSTER, MIN_PEOPLE)
fmap.save(OUT_DIR / "map.html")
print("saved map.html")
fmap

## Notes and limitations
* **Straight-line distance.** All distances are haversine (as the crow flies). Real driving distances are longer –
  often much longer near rivers, expressways or hills – so a cluster within `RADIUS_KM` in a straight line can be a longer
  drive. Travel time and traffic are not considered.
* **Outlines** are drawn from cabinet positions only. Two clusters with clear outlines can still be separated by a
  river or an expressway without a crossing nearby.
* **Cluster centre** is the simple average of the cluster's own cabinets' positions (leftovers left out), not a real
  place (it can land in a field).
* **Team size.** A full team is 3 people × 6 tickets/day (`CAP`); quiet areas get a team of 2 or 1. If teams or
  rates change, update `PEOPLE_PER_CLUSTER`, `MIN_PEOPLE`, `DEMAND_PER_PERSON` and `MIN_DEMAND_PER_PERSON` in the
  parameters cell and run the notebook again. Set `MIN_PEOPLE = 2` if nobody should work alone, and
  `SOLO_MAX_DEMAND = 6` if one person should never take more than 6.
* **Smaller teams use what is left.** Clusters are built for full teams first; only the clusters that can't reach
  12/day are tried with fewer people. Quiet areas are not rebuilt from scratch for small teams.
* `demand` is a 90-day average. Busy days will exceed `CAP`; the fill rate is a planning figure, not a guarantee.
* **Cluster IDs** (`C001`, `C002`, …) are numbered on every run, so after a data change the same area can get a
  different ID.
* **Secondary cabinets** are a list for the manager, not part of the plan: each cabinet is still visited as part of its own (primary) cluster's day unless the manager moves it.